# Phase 0 — Lesson 04: APIs & Keys
> Every AI API works the same way: send a request, get a response. The details change, the pattern doesn't.

**Source:** ai-engineering-from-scratch -> Phase 0 (Setup & Tooling), Lesson 04
**Type:** Build | **Languages:** Python, TypeScript | **Time:** ~30 minutes
**Prerequisites:** Phase 0, Lesson 01

This lesson explains *why* API keys need protection, *how* the request/response pattern works, and *what* happens when things go wrong. Run the code cells to see visualizations.


# The Big Idea: APIs Are Waiters

---


Think of an API like a waiter at a restaurant:

| Restaurant | API |
|------------|-----|
| You (customer) | Your code |
| Waiter | API endpoint |
| Kitchen | AI model server |
| Your order | Request body |
| Your meal | Response body |
| ID card to prove you can pay | API key |

You don't walk into the kitchen yourself. You give the waiter your order, the waiter takes it to the kitchen, and brings back your meal.

Your code doesn't talk to the model directly. It sends a request to the API endpoint, and the API brings back the response.


# Visual: How an API Call Works

---


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis('off')

# Boxes
boxes = [
    (0.1, 0.5, 'Your Code', '#3498db'),
    (0.4, 0.5, 'API Endpoint', '#2ecc71'),
    (0.7, 0.5, 'AI Model', '#e74c3c'),
]
for x, y, label, color in boxes:
    rect = plt.Rectangle((x, y-0.12), 0.25, 0.24, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 0.125, y, label, ha='center', va='center', color='white', fontweight='bold', fontsize=11)

# Arrows
ax.annotate('', xy=(0.37, 0.62), xytext=(0.35, 0.62),
            arrowprops=dict(arrowstyle='->', color='#3498db', lw=3))
ax.text(0.22, 0.66, 'Request\n+ API key', ha='center', fontsize=9, color='#3498db', fontweight='bold')

ax.annotate('', xy=(0.67, 0.38), xytext=(0.65, 0.38),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=3))
ax.text(0.55, 0.34, 'Response\n(JSON)', ha='center', fontsize=9, color='#e74c3c', fontweight='bold')

ax.text(0.5, 0.1, 'Every API call has 4 parts:\n1. Endpoint (URL)  2. API key  3. Request body  4. Response body',
        ha='center', fontsize=11, fontweight='bold', style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("The API Request/Response Pattern", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **The pattern never changes:** Endpoint → API Key → Request → Response. What changes is the URL, the key name, and the JSON format.


# Why API Keys Are Secrets (Not Just Passwords)

---


An API key is more than a password — it's tied to **billing**.

If someone steals your API key, they can:
- Make API calls on your account
- Run up your bill (thousands of dollars possible)
- Access your data through the API

**Never put API keys in source code.** If you commit code with a key to GitHub, bots will find it within minutes and start abusing it.

Instead, use **environment variables** — values stored outside your code, read by your program at runtime.


# Visual: Why Hardcoded Keys Are Dangerous

---


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis('off')

# Two scenarios side by side
ax.text(0.25, 0.9, 'SCENARIO A: Key in .env file', ha='center', fontsize=12, fontweight='bold', color='#27ae60')
ax.text(0.75, 0.9, 'SCENARIO B: Key hardcoded in .py', ha='center', fontsize=12, fontweight='bold', color='#e74c3c')

# Scenario A
ax.text(0.1, 0.7, '.env file:', fontsize=10, fontweight='bold')
ax.text(0.1, 0.6, 'ANTHROPIC_API_KEY=sk-ant-abc123', fontsize=9, fontfamily='monospace')
ax.text(0.1, 0.5, '.gitignore:', fontsize=10, fontweight='bold')
ax.text(0.1, 0.4, '.env', fontsize=9, fontfamily='monospace')
ax.text(0.25, 0.25, 'SAFE ✓', ha='center', fontsize=14, fontweight='bold', color='#27ae60')

# Scenario B
ax.text(0.6, 0.7, 'api_client.py:', fontsize=10, fontweight='bold')
ax.text(0.6, 0.6, 'API_KEY = "sk-ant-abc123"', fontsize=9, fontfamily='monospace')
ax.text(0.6, 0.5, 'git commit -m "Add client"', fontsize=9, fontfamily='monospace')
ax.text(0.6, 0.4, 'git push origin main', fontsize=9, fontfamily='monospace')
ax.text(0.75, 0.25, 'LEAKED ✗', ha='center', fontsize=14, fontweight='bold', color='#e74c3c')

# Arrow showing leak path
ax.annotate('', xy=(0.6, 0.35), xytext=(0.4, 0.45),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2, linestyle='--'))
ax.text(0.42, 0.5, 'Bots scan GitHub\nfor keys', fontsize=8, color='#e74c3c', ha='center')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Two Ways to Store Keys: Safe vs Dangerous", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **The fix:** Store keys in `.env` files and add `.env` to `.gitignore`. SDKs automatically read keys from environment variables.


# Step 1: Store API Keys Using Environment Variables

---


```bash
# Option A: Export in terminal (temporary, lasts until you close terminal)
export ANTHROPIC_API_KEY="sk-ant-..."
export OPENAI_API_KEY="sk-..."

# Option B: .env file (persistent, add .env to .gitignore)
cat > .env << 'EOF'
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_API_KEY=sk-...
EOF
```

**.env file example:**
```
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_API_KEY=sk-...
```

**.gitignore addition:**
```
# Environment files
.env
.env.local
```

Load the .env file in Python:
```python
from dotenv import load_dotenv
import os

load_dotenv()  # Reads .env and sets environment variables
key = os.environ.get("ANTHROPIC_API_KEY")
```


# Step 2: First API Call — Python SDK

---


The Anthropic Python SDK handles the HTTP details for you:

```python
import os
import anthropic

client = anthropic.Anthropic()  # Reads ANTHROPIC_API_KEY from env automatically

MODEL = os.environ.get("LLM_MODEL", "claude-sonnet-5")

response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[{"role": "user", "content": "What is a neural network in one sentence?"}]
)

print(response.content[0].text)
```

**What's happening:**
1. `anthropic.Anthropic()` — Creates a client, reads your API key from env
2. `client.messages.create()` — Sends the request to the API
3. `response.content[0].text` — Extracts the text from the JSON response

`LLM_MODEL` lets you switch models without changing code. Default is `claude-sonnet-5`.


# Step 3: First API Call — TypeScript

---


Phases 13-17 use TypeScript for agents and MCP servers. Same pattern, different syntax:

```typescript
import Anthropic from "@anthropic-ai/sdk";

const client = new Anthropic();

const MODEL = process.env.LLM_MODEL ?? "claude-sonnet-5";

const response = await client.messages.create({
  model: MODEL,
  max_tokens: 256,
  messages: [{ role: "user", content: "What is a neural network in one sentence?" }],
});

console.log(response.content[0].text);
```

**Key differences from Python:**
- `process.env` instead of `os.environ`
- `??` (nullish coalescing) instead of `.get()` with default
- `await` for async (TypeScript is async by default for I/O)
- Semicolons and curly braces


# Step 4: Raw HTTP — No SDK, Just Python Standard Library

---


SDKs are convenient, but they hide what's actually happening. Understanding raw HTTP helps you debug when the SDK gives a confusing error.

```python
import os
import urllib.request
import json

url = "https://api.anthropic.com/v1/messages"
headers = {
    "Content-Type": "application/json",
    "x-api-key": os.environ["ANTHROPIC_API_KEY"],
    "anthropic-version": "2023-06-01",
}
body = json.dumps({
    "model": os.environ.get("LLM_MODEL", "claude-sonnet-5"),
    "max_tokens": 256,
    "messages": [{"role": "user", "content": "What is a neural network in one sentence?"}],
}).encode()

req = urllib.request.Request(url, data=body, headers=headers, method="POST")
with urllib.request.urlopen(req) as resp:
    result = json.loads(resp.read())
    print(result["content"][0]["text"])
```

**Three headers you must send:**
- `Content-Type: application/json` — Tells the server you're sending JSON
- `x-api-key` — Your authentication key (Anthropic uses this, not `Authorization: Bearer`)
- `anthropic-version` — API version pinning (prevents breaking changes)

**This is what the SDK does under the hood.**


# Visual: SDK vs Raw HTTP — What's Under the Hood

---


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis('off')

# SDK path
ax.text(0.2, 0.8, 'SDK Path:', fontsize=11, fontweight='bold', color='#27ae60')
sdk_steps = [
    'client.messages.create()',
    'SDK builds JSON body',
    'SDK adds headers + key',
    'SDK sends HTTP request',
    'SDK parses JSON response',
    'You get Python object'
]
for i, step in enumerate(sdk_steps):
    ax.text(0.1, 0.7 - i*0.12, step, fontsize=9, fontfamily='monospace', color='#27ae60')

# Raw HTTP path
ax.text(0.6, 0.8, 'Raw HTTP Path:', fontsize=11, fontweight='bold', color='#e74c3c')
raw_steps = [
    'Build JSON body manually',
    'Set headers + x-api-key',
    'Create urllib.request.Request',
    'Call urlopen()',
    'Read response bytes',
    'Parse JSON manually'
]
for i, step in enumerate(raw_steps):
    ax.text(0.5, 0.7 - i*0.12, step, fontsize=9, fontfamily='monospace', color='#e74c3c')

ax.text(0.35, 0.05, 'Same result. SDK just hides the steps.', ha='center', fontsize=10, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("SDK vs Raw HTTP: What Gets Hidden vs What You Control", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Use SDKs for production.** Use raw HTTP when debugging — you see exactly what's sent and received.


# Rate Limits and Common Errors

---


APIs limit how many requests you can make. This prevents abuse and ensures fair usage.

| Error Code | Name | What It Means | What To Do |
|------------|------|---------------|------------|
| 401 | Unauthorized | Bad or missing API key | Check your key, check .env |
| 403 | Forbidden | Key doesn't have permission | Check your plan/billing |
| 429 | Too Many Requests | Rate limit exceeded | Wait, retry with backoff |
| 500 | Internal Server Error | Server problem | Wait, retry later |

**Rate limit strategy:** Exponential backoff — wait 1s, then 2s, then 4s, then 8s…
Never retry immediately in a tight loop.


# Visual: Exponential Backoff on Rate Limit

---


In [ ]:
import matplotlib.pyplot as plt

attempts = list(range(1, 8))
wait_times = [1, 2, 4, 8, 16, 32, 64]  # seconds

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(attempts, wait_times, color='#e74c3c', alpha=0.8)
ax.set_xlabel('Retry Attempt')
ax.set_ylabel('Wait Time (seconds)')
ax.set_title('Exponential Backoff: Wait Longer After Each Failure', fontsize=12, fontweight='bold')
ax.set_xticks(attempts)
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

for bar, t in zip(bars, wait_times):
    ax.text(bar.get_x() + bar.get_width()/2, t * 1.1, f'{t}s', ha='center', fontweight='bold')

ax.text(0.5, 0.95, 'Never retry immediately — wait longer each time', transform=ax.transAxes,
        ha='center', fontsize=10, style='italic', color='#e74c3c')

plt.tight_layout()
plt.show()


> **HTTP 429** means slow down. The API is telling you to wait. Respect it.


# Tokens: The Billing Unit

---


You're billed by **tokens**, not requests. A token is roughly 0.75 words.

| Type | What It Is | Who Pays |
|------|------------|----------|
| Input tokens | Tokens in your prompt | You |
| Output tokens | Tokens in the model's reply | You |

A long conversation costs more because every message is re-sent as input.

**Example:** "What is a neural network in one sentence?"
- Input: ~12 tokens
- Output: ~25 tokens
- Total billed: ~37 tokens


# Visual: Input vs Output Tokens

---


In [ ]:
import matplotlib.pyplot as plt

interactions = ['Simple Question', 'Code Review', 'Long Document', 'Code Generation']
input_tokens = [12, 450, 8000, 200]
output_tokens = [25, 120, 150, 800]

x = range(len(interactions))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - width/2 for i in x], input_tokens, width, label='Input (your prompt)', color='#3498db', alpha=0.8)
ax.bar([i + width/2 for i in x], output_tokens, width, label='Output (model reply)', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Tokens', fontsize=10)
ax.set_title('Token Usage by Interaction Type', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(interactions, rotation=15, ha='right')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')

for i, (inp, out) in enumerate(zip(input_tokens, output_tokens)):
    ax.text(i - width/2, inp * 1.1, f'{inp:,}', ha='center', fontsize=8, fontweight='bold')
    ax.text(i + width/2, out * 1.1, f'{out:,}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()


> **Long documents and code generation use the most tokens.** Keep prompts concise to save money.


# APIs Used in This Course

---


You don't need all of them right now. Set them up when the lesson requires it.

| API | When You Need It | Free Tier |
|-----|-----------------|-----------|
| Anthropic (Claude) | Phases 11-16 (agents, tools) | $5 credit on signup |
| OpenAI | Phase 11 (model comparison) | $5 credit on signup |
| Hugging Face | Phases 4-10 (models, datasets) | Free |

**Start with Anthropic** — it's used in the most lessons.


# Quiz Review

---


Try answering these before reading the explanations:

**Q1:** What is an API key used for when calling an LLM service?
> **Explanation:** Authenticating your identity and authorizing requests. The key identifies your account and lets the provider track usage and billing.

**Q2:** Why should API keys never be hardcoded in source code?
> **Explanation:** They can be accidentally committed to git and exposed publicly. Bots scan GitHub for keys within minutes of a push.

**Q3:** What is the recommended way to store API keys for local development?
> **Explanation:** In a `.env` file that is listed in `.gitignore`. SDKs read keys from environment variables automatically.

**Q4:** In the raw HTTP API call to Anthropic, which header carries the API key?
> **Explanation:** `x-api-key: sk-ant-...`. Anthropic uses this header, not the more common `Authorization: Bearer` pattern.

**Q5:** What happens when you exceed an API's rate limit?
> **Explanation:** The server returns HTTP 429 (Too Many Requests). You must wait before retrying, typically with exponential backoff.


# Practice Exercises

---


Do these in order:

**Exercise 1:** Get an Anthropic API key and make your first API call.

**Exercise 2:** Try the raw HTTP version and compare the response format to the SDK version.

**Exercise 3:** Intentionally use a wrong API key and read the error message.

### Sample .env file:
```
ANTHROPIC_API_KEY=sk-ant-your-key-here
OPENAI_API_KEY=sk-your-key-here
LLM_MODEL=claude-sonnet-5
```

### Sample .gitignore:
```
# Environment files (never commit these!)
.env
.env.local

# Python
__pycache__/
*.pyc
.venv/
```


# Key Terms

---


| Term | What people say | What it actually means |
|------|----------------|----------------------|
| API key | "Password for the API" | A unique string that identifies your account and authorizes requests |
| Rate limit | "They're throttling me" | Maximum requests per minute/hour to prevent abuse and ensure fair usage |
| Token | "A word" | A billing unit: input and output tokens are counted and charged separately |
| Streaming | "Real-time responses" | Getting the response word by word instead of waiting for the full response |
| .env | "Environment file" | A file storing environment variables, excluded from git via .gitignore |
| HTTP 429 | "Too many requests" | Rate limit exceeded error — wait and retry with backoff |


# Next Steps

---


When you've made your first successful API call, continue to:
**Lesson 05: Jupyter Notebooks** (you've already seen this one — it's about prototyping)
or jump to **Lesson 06: Python Environments**.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
